In [ ]:
import inspect
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd
df = pd.read_csv('retail_user_behavior_100k.csv')
df

In [ ]:
# plot a line chart of the number of unique IDs in each class with the user_action on the X-axis and the count on the Y-axis
unique_counts = df.groupby('user_action')['user_id'].nunique().reset_index()
unique_counts.columns = ['user_action', 'unique_count']

plt.figure(figsize=(10, 6))
sns.barplot(data=unique_counts, x='user_action', y='unique_count')
plt.xlabel('User Action')
plt.ylabel('Count of Unique IDs')
plt.title('Number of Unique User IDs per User Action')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
"""
Create a pivot table as a new dataframe from df. The pivot rows will be session_id, user_id, product_id, category,brand,price,channel,device_type,region,trafiic_source. The pivot columns will be `user_action` and the values will be the count of `user_action` category. Fill the missing values with 0.
"""
pivot_df = df.pivot_table(index=['session_id', 'user_id', 'product_id', 'category', 'brand', 'price', 'channel', 'device_type', 'region', 'traffic_source'],
                          columns='user_action',
                          values='event_index',
                          aggfunc='count',
                          fill_value=0)
pivot_df.reset_index(inplace=True)
pivot_df

In [ ]:
grouped_df = (
    df.groupby(
        [
            'session_id', 'user_id', 'product_id', 'category', 'brand', 'price',
            'channel', 'device_type', 'region', 'traffic_source'
        ],
        as_index=False
    ).agg(
        is_conversion=('is_conversion', 'sum'),
        drop_off_flag=('drop_off_flag', 'sum'),
        time_spent_sec=('time_spent_sec', 'sum'),
        session_length=('session_length', 'mean')
    )
)

grouped_df

In [ ]:
# Use `groupdf` if it exists; otherwise fall back to `grouped_df`
left_df = globals().get('groupdf', grouped_df)

join_keys = [
    'session_id', 'user_id', 'product_id', 'category', 'brand', 'price',
    'channel', 'device_type', 'region', 'traffic_source'
]

merged_df = left_df.merge(pivot_df, on=join_keys, how='inner')
merged_df

In [ ]:
merged_df = merged_df.drop(columns=['drop_off_flag', 'drop', 'purchase'])

In [ ]:
from sklearn.model_selection import train_test_split

# Create a stratification key combining all specified columns
stratify_col = merged_df[['category', 'brand', 'channel', 'device_type', 'region', 'traffic_source']].astype(str).agg('_'.join, axis=1)

# Make stratification safe by grouping ultra-rare strata (count < 2) into one bucket
strata_counts = stratify_col.value_counts()
stratify_safe = stratify_col.where(stratify_col.map(strata_counts) >= 2, "__RARE__")

train_df, test_df = train_test_split(
    merged_df,
    test_size=0.3,
    random_state=42,
    stratify=stratify_safe
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

In [ ]:
cols_to_drop = ['session_id', 'user_id']

train_df = train_df.drop(columns=cols_to_drop, errors='ignore')
test_df = test_df.drop(columns=cols_to_drop, errors='ignore')

print("train_df columns:", train_df.columns.tolist())
print("test_df columns:", test_df.columns.tolist())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Dependent and independent variables
target_col = "is_conversion"
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# Column groups for preprocessing
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()

# ColumnTransformer for feature preparation
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

# Decision Tree pipeline (preprocessing + model)
dt_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(random_state=42))
    ]
)

# Train and evaluate
dt_pipeline.fit(X_train, y_train)
y_pred = dt_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Extract transformed feature names from the fitted preprocessor
feature_names = dt_pipeline.named_steps["preprocessor"].get_feature_names_out()
class_names = [str(c) for c in dt_pipeline.named_steps["model"].classes_]

# Plot only the top levels to keep the diagram readable
plt.figure(figsize=(36, 18))
plot_tree(
    dt_pipeline.named_steps["model"],
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
 )
plt.title("Decision Tree (first 3 levels)")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score

# Fresh preprocessor (to avoid mutating existing dt_pipeline objects)
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced",
            oob_score=True
        ))
    ]
)

# Train
rf_pipeline.fit(X_train, y_train)

# Predict
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]
y_train_pred_rf = rf_pipeline.predict(X_train)

# Metrics
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))
print("Random Forest Train Accuracy:", accuracy_score(y_train, y_train_pred_rf))
print("Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("Random Forest OOB Score:", rf_pipeline.named_steps["model"].oob_score_)
print("\nRandom Forest Classification Report:\n")
print(classification_report(y_test, y_pred_rf))

# Confusion matrix (monitoring)
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

hgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.05, max_depth=6, max_iter=300, random_state=42
            ),
        ),
    ]
)

# HistGradientBoostingClassifier needs dense features (not sparse)
try:
    dense_ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
except TypeError:
    dense_ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

hgb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", dense_ohe, categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

hgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", hgb_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.05, max_depth=6, max_iter=300, random_state=42
            ),
        ),
    ]
)

hgb_pipeline.fit(X_train, y_train)
y_pred_hgb = hgb_pipeline.predict(X_test)
y_proba_hgb = hgb_pipeline.predict_proba(X_test)[:, 1]

print("HistGB Accuracy:", accuracy_score(y_test, y_pred_hgb))
print("HistGB ROC-AUC:", roc_auc_score(y_test, y_proba_hgb))
print(classification_report(y_test, y_pred_hgb))

In [ ]:
# Compare model performance in a pandas DataFrame
model_reports = {
    "Decision Tree": classification_report(y_test, y_pred, output_dict=True, zero_division=0),
    "Random Forest": classification_report(y_test, y_pred_rf, output_dict=True, zero_division=0),
    "HistGradientBoosting": classification_report(y_test, y_pred_hgb, output_dict=True, zero_division=0),
}

model_comparison_df = pd.DataFrame(
    [
        {
            "Model": "Decision Tree",
            "Precision": model_reports["Decision Tree"]["weighted avg"]["precision"],
            "Recall": model_reports["Decision Tree"]["weighted avg"]["recall"],
            "F1-Score": model_reports["Decision Tree"]["weighted avg"]["f1-score"],
        },
        {
            "Model": "Random Forest",
            "Precision": model_reports["Random Forest"]["weighted avg"]["precision"],
            "Recall": model_reports["Random Forest"]["weighted avg"]["recall"],
            "F1-Score": model_reports["Random Forest"]["weighted avg"]["f1-score"],
        },
        {
            "Model": "HistGradientBoosting",
            "Precision": model_reports["HistGradientBoosting"]["weighted avg"]["precision"],
            "Recall": model_reports["HistGradientBoosting"]["weighted avg"]["recall"],
            "F1-Score": model_reports["HistGradientBoosting"]["weighted avg"]["f1-score"],
        },
    ]
)

model_comparison_df = model_comparison_df.round(4)
display(model_comparison_df)

In [ ]:
# Metrics on X-axis, models as grouped bars
plot_df = model_comparison_df.set_index("Model").T  # rows: metrics, cols: models

ax = plot_df.plot(
    kind="bar",
    figsize=(10, 6),
    rot=0
)

plt.title("Model Comparison by Metric")
plt.xlabel("Metric")
plt.ylabel("Score")

# Zoom Y-axis around actual values for better visual separation
y_min = plot_df.min().min()
y_max = plot_df.max().max()
padding = max((y_max - y_min) * 0.15, 0.005)
lower = max(0, y_min - padding)
upper = min(1, y_max + padding)
plt.ylim(lower, upper)

plt.legend(title="Model")
plt.tight_layout()
plt.show()